# 👤 Dataset 5: Adult Census Income
**Propósito:** Clasificación binaria — Predecir si una persona gana >$50K/año.  
**Fuente:** Censo de EE. UU. 1994, extraído por Barry Becker (UCI Repository).  
> 📌 Dataset clásico para estudiar **Fairness in ML** (sesgo por `race` y `sex`).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
print("✅ Librerías cargadas")

## 1. Carga de Datos
> El archivo no tiene encabezado — asignamos nombres manualmente según la documentación UCI.

In [ ]:
cols = ['age','workclass','fnlwgt','education','education-num',
        'marital-status','occupation','relationship','race','sex',
        'capital-gain','capital-loss','hours-per-week','native-country','income']

train = pd.read_csv('../datasets/5 data sets/dataset5/adult.data', 
                    header=None, names=cols, na_values=' ?', skipinitialspace=True)
test  = pd.read_csv('../datasets/5 data sets/dataset5/adult.test',
                    header=None, names=cols, na_values=' ?', skipinitialspace=True,
                    skiprows=1)

# El archivo test tiene un '.' al final de los labels, limpiar
test['income'] = test['income'].str.replace('.','', regex=False)

df = pd.concat([train, test], ignore_index=True)
print(f"Train: {train.shape}  Test: {test.shape}  Total: {df.shape}")
df.head(3)

## 2. Análisis de Valores Nulos (el '?' ya fue reemplazado por NaN)

In [ ]:
nulos = df.isnull().sum()
nulos_pct = (nulos/len(df)*100).round(2)
resumen = pd.DataFrame({'Nulos': nulos, '%': nulos_pct}).query('Nulos > 0')
print(resumen)

plt.figure(figsize=(7,3))
resumen['%'].plot(kind='bar', color='tomato')
plt.title('Columnas con valores faltantes (? → NaN)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('adult_nulos.png', dpi=100)
plt.show()

## 3. Limpieza

In [ ]:
# Imputar con moda (todas son categóricas)
for col in ['workclass','occupation','native-country']:
    df[col] = df[col].fillna(df[col].mode()[0])

# Limpiar espacios en labels
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip()

print(f"Nulos restantes: {df.isnull().sum().sum()}")
print(f"Distribución de clases:")
print(df['income'].value_counts())

## 4. Análisis de Sesgo Algorítmico (Fairness)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Ingresos por género
sex_income = df.groupby(['sex','income']).size().unstack(fill_value=0)
sex_income.div(sex_income.sum(axis=1), axis=0).plot(kind='bar', ax=axes[0],
    color=['steelblue','tomato'])
axes[0].set_title('Proporción de ingresos por Género')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(['<=50K', '>50K'])

# Ingresos por raza
race_income = df.groupby(['race','income']).size().unstack(fill_value=0)
race_income.div(race_income.sum(axis=1), axis=0)['>50K'].sort_values().plot(
    kind='barh', ax=axes[1], color='coral')
axes[1].set_title('% que gana >50K por Raza')
axes[1].set_xlabel('Proporción')

plt.tight_layout()
plt.savefig('adult_fairness.png', dpi=100)
plt.show()

## 5. EDA — Variables Numéricas

In [ ]:
fig, axes = plt.subplots(1,3, figsize=(15,4))
df['age'].hist(ax=axes[0], bins=30, color='steelblue', edgecolor='white', by=None)
axes[0].set_title('Distribución de Edad')

df['hours-per-week'].hist(ax=axes[1], bins=30, color='seagreen', edgecolor='white')
axes[1].set_title('Horas de trabajo por semana')

df['education-num'].value_counts().sort_index().plot(kind='bar', ax=axes[2], color='purple')
axes[2].set_title('Años de educación')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('adult_eda.png', dpi=100)
plt.show()

## 6. Preparación del Modelo

In [ ]:
from sklearn.preprocessing import LabelEncoder
df_model = df.copy()

# Codificar target
df_model['TARGET'] = (df_model['income'] == '>50K').astype(int)
df_model = df_model.drop(columns=['income','fnlwgt'])  # fnlwgt = peso estadístico, no predictivo

# Encoding
le = LabelEncoder()
cat_cols = df_model.select_dtypes(include='object').columns
for col in cat_cols:
    df_model[col] = le.fit_transform(df_model[col].astype(str))

X = df_model.drop(columns=['TARGET'])
y = df_model['TARGET']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train: {X_train.shape[0]}  Test: {X_test.shape[0]}")
print(f"Prevalencia >50K: {y.mean():.1%}")

## 7. Modelo — Regresión Logística + Random Forest

In [ ]:
# Escalar para Logistic Regression
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# Logistic Regression
lr = LogisticRegression(max_iter=500, random_state=42)
lr.fit(X_train_s, y_train)
lr_pred = lr.predict(X_test_s)
lr_prob = lr.predict_proba(X_test_s)[:,1]

print("=== REGRESIÓN LOGÍSTICA ===")
print(classification_report(y_test, lr_pred, target_names=['<=50K','>50K']))
print(f"ROC-AUC: {roc_auc_score(y_test, lr_prob):.4f}")

In [ ]:
# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:,1]

print("=== RANDOM FOREST ===")
print(classification_report(y_test, rf_pred, target_names=['<=50K','>50K']))
print(f"ROC-AUC: {roc_auc_score(y_test, rf_prob):.4f}")

# Feature importances
imp = pd.Series(rf.feature_importances_, index=X.columns).nlargest(10).sort_values()
imp.plot(kind='barh', figsize=(8,5), color='steelblue')
plt.title('Top 10 Features — Random Forest')
plt.tight_layout()
plt.savefig('adult_importancias.png', dpi=100)
plt.show()

## ✅ Resumen
| Métrica | Log. Reg. | Random Forest |
|---|---|---|
| ROC-AUC | ~0.89 | ~0.92 |
| Accuracy | ~85% | ~87% |

**Hallazgos de Fairness:**
- Los hombres tienen ~3x más probabilidad de ganar >50K que las mujeres
- Hay disparidades significativas por raza que el modelo puede perpetuar
- Para uso en producción se recomienda aplicar técnicas de Fairness como reweighting o adversarial debiasing